# FLUX Phase 1: Continuous Semantic Encoder (CSE)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Replace the tokenizer + embedding layer with a continuous wave-based encoder. No vocabulary, no tokens — just physics.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Smoke Test** — Verify CSE builds and gradients flow
3. **Train** — Reconstruction + semantic contrastive learning (~5000 steps)
4. **Upload** — Checkpoint → HuggingFace Hub
5. **Test 1** — Reconstruction quality (loss < 0.1)
6. **Test 2** — Language-agnostic encoding (UTF-8, cross-lingual)
7. **Test 3** — Semantic proximity ordering (18/20 triplets)
8. **Demo 1** — Wave component visualization
9. **Demo 2** — Interference patterns (constructive/destructive)
10. **Final** — Upload logs → HuggingFace & push to GitHub

### Logging:
Every cell writes to `logs/phase1.log`. The log is saved to HuggingFace and GitHub after each major step.

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("\u2139 Repo already exists \u2014 pulling latest changes...")
    result = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=str(WORK_DIR), capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print("\u2713 Pulled latest")
else:
    print(f"\u2139 Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("\u2713 Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")

---
## Cell 2: Install Dependencies & Setup Structure

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys
import torch

sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase1'))

from flux_utils import (
    PhaseLogger, get_device, get_hardware_info,
    upload_checkpoint_to_hf, upload_logs_to_hf,
    git_commit_and_push, _resolve_hf_token,
)

# \u2500\u2500 Initialize Phase 1 Logger \u2500\u2500
log = PhaseLogger(phase=1)
log.separator("Phase 1: Continuous Semantic Encoder")
log.cell_start("Cell 3 \u2014 Hardware & Secrets")

# \u2500\u2500 Detect hardware \u2500\u2500
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# \u2500\u2500 Load HuggingFace token from Kaggle secrets \u2500\u2500
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
else:
    log.warning("No HuggingFace token \u2014 checkpoint upload will be skipped")
    log.warning("Add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 \u2014 Hardware & Secrets")

---
## Cell 4: Smoke Test — Verify CSE Builds & Gradients Flow

In [ ]:
from phases.phase1.cse import ContinuousSemanticEncoder, WaveDecoder
from phases.phase1.wave_types import TOTAL_WAVE_DIM, WAVE_DIMS

log.cell_start("Cell 4 \u2014 Smoke Test")

# Build models
cse = ContinuousSemanticEncoder(device=DEVICE).to(DEVICE)
decoder = WaveDecoder(wave_dim=TOTAL_WAVE_DIM).to(DEVICE)

param_count = sum(p.numel() for p in cse.parameters())
dec_count = sum(p.numel() for p in decoder.parameters())
log.metric("CSE parameters", f"{param_count:,}")
log.metric("Decoder parameters", f"{dec_count:,}")
log.metric("Total parameters", f"{param_count + dec_count:,}")

# Test encoding diverse inputs
test_inputs = [
    ('English',  'Hello, world!'),
    ('Short',    'dog'),
    ('Chinese',  '\u4f60\u597d\u4e16\u754c'),
    ('Code',     'def f(): pass'),
    ('Single',   'a'),
    ('Empty',    ''),
]

all_ok = True
for label, text in test_inputs:
    try:
        with torch.no_grad():
            wave = cse.encode(text)
        log.success(f"{label}: \"{text}\" \u2192 [{wave.seq_len}, {wave.full.shape[-1]}]")
    except Exception as e:
        log.error(f"{label}: \"{text}\" \u2192 FAILED: {e}")
        all_ok = False

# Test backward pass
wave = cse.encode('gradient flow test')
logits = decoder(wave.full)
target = cse.text_to_bytes('gradient flow test')
loss = torch.nn.functional.cross_entropy(logits[:len(target)], target)
loss.backward()
log.success(f"Backward pass OK \u2014 loss = {loss.item():.4f}")

# Cleanup
del cse, decoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None

if all_ok:
    log.success("All smoke tests passed")
else:
    log.error("Some smoke tests failed \u2014 check above")

log.cell_end("Cell 4 \u2014 Smoke Test", "PASS" if all_ok else "FAIL")

---
## Cell 5: Train the CSE

**Training objectives:**
- Reconstruction: encode \u2192 wave \u2192 decode back to bytes
- Semantic contrastive: similar words \u2192 similar waves
- Antonym separation: opposite words \u2192 low similarity

~5000 steps \u2022 ~10 min on GPU \u2022 ~60 min on CPU

In [ ]:
import os
log.cell_start("Cell 5 \u2014 Training")
log.info(f"Training with device={DEVICE}, steps=5000")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python train_cse.py --steps 5000 --device {DEVICE}

# Verify checkpoint
ckpt_path = WORK_DIR / 'checkpoints' / 'phase1.phase.pt'
if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / 1e6
    log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
else:
    log.error("Checkpoint NOT found \u2014 training may have failed")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 5 \u2014 Training")

---
## Cell 6: Upload Checkpoint to HuggingFace Hub
Saves the trained model to `https://huggingface.co/UnseenGAP/FLUX`

In [ ]:
log.cell_start("Cell 6 \u2014 HuggingFace Upload")

uploaded = upload_checkpoint_to_hf(phase=1, hf_token=HF_TOKEN)
if uploaded:
    log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
else:
    log.warning("Checkpoint upload skipped \u2014 check HF_TOKEN")

# Upload log so far
upload_logs_to_hf(phase=1, hf_token=HF_TOKEN)

log.cell_end("Cell 6 \u2014 HuggingFace Upload")

---
## Cell 7: Test 1 — Reconstruction Quality

**Pass criteria:**
- Average reconstruction loss < 0.1 (byte accuracy > 90%)
- No single sentence has loss > 0.5
- Runs in < 60 seconds

In [ ]:
log.cell_start("Cell 7 \u2014 Test 1: Reconstruction")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python test_phase1_test1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 7 \u2014 Test 1: Reconstruction")

---
## Cell 8: Test 2 — Language-Agnostic Encoding

**Pass criteria:**
- Encodes any UTF-8 input without errors
- Cross-language similarity > 0.4 for 4/5 pairs
- Encoding is deterministic

In [ ]:
log.cell_start("Cell 8 \u2014 Test 2: Language Agnostic")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python test_phase1_test2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 8 \u2014 Test 2: Language Agnostic")

---
## Cell 9: Test 3 — Semantic Proximity Ordering

**Pass criteria:**
- sim("dog", "cat") > sim("dog", "democracy") for 20 triplets
- 18/20 must pass

In [ ]:
log.cell_start("Cell 9 \u2014 Test 3: Semantic Proximity")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python test_phase1_test3.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 9 \u2014 Test 3: Semantic Proximity")

---
## Cell 10: Demo 1 — Semantic Wave Visualization
Heatmap of all 5 wave components (phonetic, syntactic, semantic, temporal, intensity) for sample sentences.

In [ ]:
log.cell_start("Cell 10 \u2014 Demo 1: Wave Visualization")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python demo_phase1_demo1.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase1' / 'demo1_wave_visualization.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("Wave visualization generated")
else:
    log.warning("Visualization not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 10 \u2014 Demo 1: Wave Visualization")

---
## Cell 11: Demo 2 — Interference Patterns
Constructive (similar), destructive (opposite), and neutral (unrelated) interference.

In [ ]:
log.cell_start("Cell 11 \u2014 Demo 2: Interference Patterns")

os.chdir(str(WORK_DIR / 'phases' / 'phase1'))
!python demo_phase1_demo2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 11 \u2014 Demo 2: Interference Patterns")

---
## Cell 12: Interactive Exploration
Try encoding your own text and comparing similarities.

In [ ]:
import torch
import torch.nn.functional as F
from phases.phase1.cse import ContinuousSemanticEncoder
from flux_utils import load_checkpoint

log.cell_start("Cell 12 \u2014 Interactive Exploration")

checkpoint = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**checkpoint['config'], device='cpu')
cse.load_state_dict(checkpoint['state_dict'])
cse.eval()

def similarity(text1: str, text2: str) -> float:
    """Compute semantic similarity between two texts."""
    with torch.no_grad():
        w1 = cse.encode(text1)
        w2 = cse.encode(text2)
    v1 = w1.semantic.mean(dim=0)
    v2 = w2.semantic.mean(dim=0)
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

# \u2500\u2500 Try your own pairs! Edit this list: \u2500\u2500
pairs = [
    ("dog", "cat"),
    ("dog", "democracy"),
    ("king", "queen"),
    ("hot", "cold"),
    ("hello", "bonjour"),
    ("python code", "javascript code"),
    ("I love pizza", "Pizza is my favorite food"),
    ("the sun is a star", "le soleil est une \u00e9toile"),
]

print(f"{'Text 1':<35} {'Text 2':<35} {'Similarity':>10}")
print("\u2500" * 82)
for t1, t2 in pairs:
    sim = similarity(t1, t2)
    log.metric(f"sim(\"{t1}\", \"{t2}\")", f"{sim:.4f}")
    print(f"{t1:<35} {t2:<35} {sim:>10.4f}")

del cse, checkpoint
log.cell_end("Cell 12 \u2014 Interactive Exploration")

---
## Cell 13: View Results Summary

In [ ]:
log.cell_start("Cell 13 \u2014 Results Summary")

results_path = WORK_DIR / 'phases' / 'phase1' / 'RESULTS_PHASE_1.md'
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    print('Results file not yet generated \u2014 run tests first')
    log.warning("No results file found")

log.cell_end("Cell 13 \u2014 Results Summary")

---
## Cell 14: View Full Phase Log

In [ ]:
print("\n" + "=" * 60)
print("FULL PHASE 1 LOG")
print("=" * 60)
print(log.get_log_contents())

---
## Cell 15: Final Upload — Logs to HuggingFace & GitHub
Pushes the complete log and results back to both platforms.

In [ ]:
log.cell_start("Cell 15 \u2014 Final Upload")

# \u2500\u2500 Upload final log to HuggingFace \u2500\u2500
upload_logs_to_hf(phase=1, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

# \u2500\u2500 Commit logs + results to GitHub \u2500\u2500
git_commit_and_push(
    files=['logs/phase1.log', 'results/', 'phases/phase1/RESULTS_PHASE_1.md'],
    message='Phase 1: training logs + results [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 15 \u2014 Final Upload")

print("\n" + "=" * 60)
print("\u2713 PHASE 1 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 2 \u2014 Resonance Field Core")
print("=" * 60)

---
## Cell 16: Save Artifacts to Kaggle Output

In [ ]:
import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# Copy checkpoint
ckpt = WORK_DIR / 'checkpoints' / 'phase1.phase.pt'
if ckpt.exists():
    shutil.copy(ckpt, output_dir / 'phase1.phase.pt')
    print('\u2713 Checkpoint \u2192 output/')

# Copy results
for f in (WORK_DIR / 'phases' / 'phase1').glob('RESULTS_*'):
    shutil.copy(f, output_dir / f.name)
    print(f'\u2713 {f.name} \u2192 output/')

# Copy log
log_f = WORK_DIR / 'logs' / 'phase1.log'
if log_f.exists():
    shutil.copy(log_f, output_dir / 'phase1.log')
    print('\u2713 phase1.log \u2192 output/')

# Copy visualization
viz = WORK_DIR / 'phases' / 'phase1' / 'demo1_wave_visualization.png'
if viz.exists():
    shutil.copy(viz, output_dir / viz.name)
    print('\u2713 Visualization \u2192 output/')

print(f'\nOutput artifacts:')
for f in sorted(output_dir.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

---
## Phase 1 Acceptance Criteria

| Criteria | Test | Target |
|---|---|---|
| Any UTF-8 encodes without errors | Test 2 | All inputs |
| Similar words high similarity | Test 3 | > 0.5 avg |
| Semantic proximity ordering | Test 3 | \u2265 18/20 |
| Encoding is deterministic | Test 2 | max_diff < 1e-6 |
| Reconstruction loss < 0.1 | Test 1 | accuracy > 90% |
| Cross-language similarity | Test 2 | > 0.4 for 4/5 |
| Checkpoint on HuggingFace | Cell 6 | \u2713 |
| Logs saved | Cell 15 | \u2713 |

**When all criteria pass \u2192 Phase 1 complete \u2192 Phase 2 unlocked!**